In [ ]:
# Plot 3: Individual Scenario Overviews (Grid Plot)
num_scenarios = len(scenarios_data)
cols = 2
rows = (num_scenarios + 1) // cols

fig, axes = plt.subplots(rows, cols, figsize=(15, 4 * rows))
axes = axes.flatten()

for i, (name, data) in enumerate(sorted(scenarios_data.items())):
    ax = axes[i]
    ax.plot(data['load_agg'], label='Load', color='blue')
    ax.plot(data['solar_agg'], label='Solar', color='orange')
    ax.plot(data['net_load'], label='Net Load', color='green', linestyle='--')
    ax.set_title(name)
    ax.legend()
    ax.set_ylabel('Power (kW)')
    # Format x-axis to show hours only for cleanliness
    # ax.xaxis.set_major_formatter(plt.dates.DateFormatter('%H:%M'))

# Hide empty subplots
for i in range(len(scenarios_data), len(axes)):
    fig.delaxes(axes[i])

plt.tight_layout()
plt.show()

In [ ]:
# Plot 2: Compare Aggregate Solar Profiles
plt.figure(figsize=(14, 8))
for name in scenarios_data:
    plt.plot(scenarios_data[name]['solar_agg'], label=name, alpha=0.7)

plt.title('Aggregate Solar Profiles (10 Houses)')
plt.ylabel('Power (kW)')
plt.xlabel('Time')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
# Plot 1: Compare Aggregate Load Profiles
plt.figure(figsize=(14, 8))
for name in scenarios_data:
    plt.plot(scenarios_data[name]['load_agg'], label=name, alpha=0.7)

plt.title('Aggregate Load Profiles (10 Houses)')
plt.ylabel('Power (kW)')
plt.xlabel('Time')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
# Calculate Aggregates (Sum of all houses)
scenario_stats = []

for name, data in scenarios_data.items():
    if 'load' in data and 'solar' in data:
        load_agg = data['load'].sum(axis=1) # Sum across columns (houses)
        solar_agg = data['solar'].sum(axis=1)
        net_load = load_agg - solar_agg
        
        scenarios_data[name]['load_agg'] = load_agg
        scenarios_data[name]['solar_agg'] = solar_agg
        scenarios_data[name]['net_load'] = net_load
        
        scenario_stats.append({
            'Scenario': name,
            'Total Daily Load (kWh)': load_agg.sum() / 4, # divide by 4 for 15-min intervals
            'Total Daily Solar (kWh)': solar_agg.sum() / 4,
            'Peak Load (kW)': load_agg.max(),
            'Peak Solar (kW)': solar_agg.max()
        })

stats_df = pd.DataFrame(scenario_stats).set_index('Scenario')
display(stats_df)

In [ ]:
def load_scenarios(data_dir):
    """
    Parses files in the directory and organizes them by scenario.
    Returns a dictionary: scenarios[scenario_name] = {'load': df, 'solar': df}
    """
    scenarios = {}
    files = glob.glob(os.path.join(data_dir, '*.csv'))
    
    for file_path in files:
        filename = os.path.basename(file_path)
        
        # files are named like 'load_scenario_name.csv' or 'solar_scenario_name.csv'
        if filename.startswith('load_'):
            type_ key = 'load'
            scenario_name = filename.replace('load_', '').replace('.csv', '')
        elif filename.startswith('solar_'):
            type_key = 'solar'
            scenario_name = filename.replace('solar_', '').replace('.csv', '')
        else:
            continue
            
        # Read Data
        # Assuming no header for time, and 10 columns for houses
        df = pd.read_csv(file_path)
        
        # Create 15-min time index for 24 hours (96 steps)
        # Using a dummy date to make plotting easier
        times = pd.date_range(start='2024-01-01', periods=96, freq='15min')
        df.index = times
        
        if scenario_name not in scenarios:
            scenarios[scenario_name] = {}
        
        scenarios[scenario_name][type_key] = df
        
    return scenarios

scenarios_data = load_scenarios(DATA_DIR)
print(f"Loaded {len(scenarios_data)} scenarios: {list(scenarios_data.keys())}")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import os
import glob
import numpy as np

# Set style
plt.style.use('ggplot')
plt.rcParams['figure.figsize'] = (12, 6)

# Define path
DATA_DIR = os.path.join('data', 'forecast_scenarios')

# List files
files = glob.glob(os.path.join(DATA_DIR, '*.csv'))
print(f"Found {len(files)} files.")
for f in files[:5]:
    print(os.path.basename(f))
print("...")

# Scenario Analysis
This notebook analyzes the forecast scenarios located in `data/forecast_scenarios`. It loads the Load and Solar profiles for each scenario and plots them to visualise the differences.